In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

RANDOM_STATE = 42

fichier_csv = "dataset_ml_clean.csv"
print(f"Chargement des données depuis {fichier_csv}...\n")

data = pd.read_csv(fichier_csv, sep=';')

data['score'] = data['score'].apply(lambda x: x / 100.0 if x > 10 else x)

print("Exemple de scores corrigés :")
print(data[['titre', 'score']].head(), "\n")

cat_variables = ['saison', 'studio', 'genre', 'rating', 'source']
df = pd.get_dummies(data=data, columns=cat_variables, drop_first=True)

cols_to_drop = ['anime_id', 'titre', 'score']
features = [x for x in df.columns if x not in cols_to_drop]

X = df[features]
y = df['score']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=RANDOM_STATE)

print("Démarrage de l'entraînement du modèle avec Early Stopping...\n")

xgb_model = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    eval_metric="rmse",
    early_stopping_rounds=15
)

eval_set = [(X_train, y_train), (X_test, y_test)]

xgb_model.fit(
    X_train,
    y_train,
    eval_set=eval_set,
    verbose=50
)

print("\n" + "="*40)
print("--- MÉTRIQUES FINALES ---")
print("="*40)

predictions_train = xgb_model.predict(X_train)
predictions_test = xgb_model.predict(X_test)

print(f"R2 Score (Entraînement) : {r2_score(y_train, predictions_train):.4f} (Proche de 1 = Parfait)")
print(f"R2 Score (Test)         : {r2_score(y_test, predictions_test):.4f} (Performance réelle sur données inconnues)")
print(f"Erreur moyenne (RMSE)   : {mean_squared_error(y_test, predictions_test) ** 0.5:.4f} points d'écart")

resultats = X_test.copy()
resultats['anime_id'] = data.loc[X_test.index, 'anime_id']
resultats['vrai_score'] = y_test
resultats['score_predit_XGBoost'] = predictions_test

export_sql = resultats[['anime_id', 'score_predit_XGBoost']]
nom_fichier_export = "predictions_a_importer.csv"
export_sql.to_csv(nom_fichier_export, index=False)

print(f"\nExport terminé : '{nom_fichier_export}'.")
print("Fichier prêt à être réintégré dans SQL Server pour Power BI !")

Chargement des données depuis dataset_ml_clean.csv...

Exemple de scores corrigés :
       titre  score
0  Bus Gamer   6.46
1     Miyuki   6.74
2    Etotama   6.79
3    Nyanbo!   6.36
4    Urahara   5.83 

Démarrage de l'entraînement du modèle avec Early Stopping...

[0]	validation_0-rmse:0.75802	validation_1-rmse:0.74723
[50]	validation_0-rmse:0.32683	validation_1-rmse:0.46008
[100]	validation_0-rmse:0.28237	validation_1-rmse:0.44839
[150]	validation_0-rmse:0.27160	validation_1-rmse:0.44576
[200]	validation_0-rmse:0.26438	validation_1-rmse:0.44381
[250]	validation_0-rmse:0.25752	validation_1-rmse:0.44241
[300]	validation_0-rmse:0.25124	validation_1-rmse:0.44122
[350]	validation_0-rmse:0.24545	validation_1-rmse:0.44022
[400]	validation_0-rmse:0.23983	validation_1-rmse:0.43964
[412]	validation_0-rmse:0.23848	validation_1-rmse:0.43962

--- MÉTRIQUES FINALES ---
R2 Score (Entraînement) : 0.9063 (Proche de 1 = Parfait)
R2 Score (Test)         : 0.6754 (Performance réelle sur données inconn

In [ ]:
scores_complets = xgb_model.predict(X)

df_predictions_completes = pd.DataFrame({
    'anime_id': df['anime_id'],
    'score_predit_XGBoost': scores_complets
})

df_predictions_completes.to_csv('predictions_completes.csv', index=False)

print(f"Export terminé ! Nombre total de prédictions : {len(df_predictions_completes)}")

Export terminé ! Nombre total de prédictions : 5015
